# Figure 13.4: Kernel positioned at X[0,3]

Adjust the kernel size and values, then step through positions to see how each output value is computed from the element-wise product of the kernel and the local input patch.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display


def make_input(N, mode='random', seed=42):
    rng = np.random.default_rng(seed)
    if mode == 'random':
        return rng.integers(0, 10, size=(N, N)).astype(float)
    elif mode == 'ramp':
        return np.arange(1, N * N + 1, dtype=float).reshape(N, N)
    else:  # checkerboard
        i, j = np.indices((N, N))
        return ((i + j) % 2).astype(float)


KERNEL_PRESETS = {
    'identity': lambda F: np.eye(F) if F == 1 else (lambda k: (k.__setitem__((F//2, F//2), 1), k)[1])(np.zeros((F, F))),
    'edge':     lambda F: (lambda k: (k.__setitem__((F//2, F//2), F*F-1), k)[1])(np.full((F, F), -1.0)),
    'blur':     lambda F: np.full((F, F), 1.0 / (F * F)),
}


def draw_conv(N=7, F=3, step=0, input_mode='random', kernel_preset='identity'):
    X = make_input(N, input_mode)
    O = N - F + 1
    total_pos = O * O
    step = min(step, total_pos - 1)

    K = KERNEL_PRESETS.get(kernel_preset, KERNEL_PRESETS['identity'])(F)

    ri, ci = divmod(step, O)

    # Build output so far
    Y = np.full((O, O), np.nan)
    for pp in range(step + 1):
        r, c = divmod(pp, O)
        Y[r, c] = np.sum(X[r:r+F, c:c+F] * K)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

    # --- Input ---
    ax = axes[0]
    ax.imshow(np.zeros((N, N)), cmap='Blues', vmin=0, vmax=2, aspect='equal')
    for ii in range(N):
        for jj in range(N):
            in_win = ri <= ii < ri + F and ci <= jj < ci + F
            c_face = '#2563eb' if in_win else '#e2e8f0'
            c_text = 'white' if in_win else '#0f172a'
            rect = patches.FancyBboxPatch((jj - 0.45, ii - 0.45), 0.9, 0.9,
                                          boxstyle='round,pad=0.05', fc=c_face, ec='white', lw=1)
            ax.add_patch(rect)
            ax.text(jj, ii, str(int(X[ii, jj])), ha='center', va='center',
                    fontsize=11, fontweight='bold', color=c_text)
    ax.set_xlim(-0.5, N - 0.5); ax.set_ylim(N - 0.5, -0.5)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'Input X  ({N}×{N})', fontsize=12)

    # --- Kernel ---
    ax = axes[1]
    for ii in range(F):
        for jj in range(F):
            rect = patches.FancyBboxPatch((jj - 0.45, ii - 0.45), 0.9, 0.9,
                                          boxstyle='round,pad=0.05', fc='#fbbf24', ec='white', lw=1)
            ax.add_patch(rect)
            ax.text(jj, ii, str(round(K[ii, jj], 3)), ha='center', va='center',
                    fontsize=11, fontweight='bold', color='#0f172a')
    ax.set_xlim(-0.5, F - 0.5); ax.set_ylim(F - 0.5, -0.5)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'Kernel W  ({F}×{F})', fontsize=12)

    # --- Output ---
    ax = axes[2]
    yn = np.nan_to_num(Y, nan=0)
    mn, mx = (np.nanmin(Y), np.nanmax(Y)) if not np.all(np.isnan(Y)) else (0, 1)
    for ii in range(O):
        for jj in range(O):
            is_cur = ii == ri and jj == ci
            v = Y[ii, jj]
            c_face = '#059669' if is_cur else ('#a7f3d0' if not np.isnan(v) else '#f1f5f9')
            c_text = 'white' if is_cur else '#0f172a'
            rect = patches.FancyBboxPatch((jj - 0.45, ii - 0.45), 0.9, 0.9,
                                          boxstyle='round,pad=0.05', fc=c_face, ec='white', lw=1)
            ax.add_patch(rect)
            if not np.isnan(v):
                ax.text(jj, ii, str(round(v, 2)), ha='center', va='center',
                        fontsize=10, fontweight='bold', color=c_text)
    ax.set_xlim(-0.5, O - 0.5); ax.set_ylim(O - 0.5, -0.5)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'Output Y  ({O}×{O}) — step {step+1}/{total_pos}', fontsize=12)

    current_val = np.sum(X[ri:ri+F, ci:ci+F] * K)
    fig.suptitle(
        f'Y[{ri}][{ci}] = Σ(X[{ri}:{ri+F}, {ci}:{ci+F}] ⊙ W) = {round(current_val, 3)}',
        fontsize=11, color='#059669', y=0.02
    )
    plt.tight_layout()
    plt.show()


N_slider = widgets.IntSlider(min=5, max=9, step=1, value=7, description='Input N×N', continuous_update=False)
F_slider = widgets.IntSlider(min=1, max=5, step=2, value=3, description='Kernel F×F', continuous_update=False)
step_slider = widgets.IntSlider(min=0, max=24, step=1, value=0, description='Step', continuous_update=False)
input_drop = widgets.Dropdown(options=['random', 'ramp', 'checker'], value='random', description='Input')
kernel_drop = widgets.Dropdown(options=['identity', 'edge', 'blur'], value='identity', description='Kernel')

def update_step_max(change=None):
    N, F = N_slider.value, F_slider.value
    O = N - F + 1
    step_slider.max = O * O - 1
    step_slider.value = min(step_slider.value, step_slider.max)

N_slider.observe(update_step_max, names='value')
F_slider.observe(update_step_max, names='value')

widgets.interact(
    draw_conv,
    N=N_slider, F=F_slider, step=step_slider,
    input_mode=input_drop, kernel_preset=kernel_drop,
)